In [340]:
import os
import pandas as pd
import numpy as np
import matplotlib
import sklearn
print(os.getcwd())

EDIFICIO_NAME = 'Edificio1.csv'

ASCENDING_SPEED = 1.0
DESCENDING_SPEED = 2.0
HORIZONTAL_SPEED = 1.5

ASCENDING_ENERGY= 50
DESCENDING_ENERGY= 5
HORIZONTAL_ENERGY = 10

BATTERY_ONE = 3600  # 1 Wh = 3600 J
BATTERY_TWO = 21600 # 6 Wh = 21600 J

/home/michele/big-project-FOR


# Reading CSV and creating pairs


In [341]:
edificio = pd.read_csv(EDIFICIO_NAME)

edificio_pairs = edificio.rename_axis('index1').reset_index().merge(edificio.rename_axis('index2').reset_index(), how='cross')

edificio_pairs = edificio_pairs[edificio_pairs['index1'] != edificio_pairs['index2']] #keeping only pairs with different points

In [342]:
first_point_cols= ['x_x','y_x','z_x']
second_point_cols=['x_y','y_y','z_y']
diff_cols= ['dx','dy','dz']

# Computing distances 

In [343]:
def process_edificio(dataframe):
    #distances and deltas on coordinates
    dataframe[diff_cols] = (dataframe[second_point_cols].values - dataframe[first_point_cols].values)
    dataframe['euclidean_distance'] = np.sqrt((dataframe[diff_cols]**2).sum(axis=1)) #axis=1 sum on columns 
    
    
    # conditions
    condition_near = (dataframe['euclidean_distance'] <= 4.0)
    
    
    condition_tunnel = (
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dy'].abs() <= 0.5)) | 
        ((dataframe['dy'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5)) | 
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5))
    )
    condition_tunnel_and_not_far = (dataframe['euclidean_distance'] <= 11.0) & condition_tunnel
    
    # final connection condition
    dataframe["are_connected"] = condition_near | condition_tunnel_and_not_far
    return dataframe



In [344]:
edificio_pairs = process_edificio(edificio_pairs)

edificio_pairs.head()
print("\nSHAPE", edificio_pairs.shape)



SHAPE (4032, 13)


In [345]:
edificio_pairs["d_horizontal"] = np.sqrt(edificio_pairs['dx']**2 + edificio_pairs['dy']**2)
edificio_pairs["d_vertical"] = edificio_pairs['dz']

edificio_pairs.drop(columns=['dx','dy','dz'], inplace=True)

edificio_pairs.head(10)

,index1,x_x,y_x,z_x,index2,x_y,y_y,z_y,euclidean_distance,are_connected,d_horizontal,d_vertical
1,0,2.5,3.25,0.0,1,-6.0,3.25,0.0,8.500000,True,8.500000,0.0
2,0,2.5,3.25,0.0,2,-6.0,11.75,0.0,12.020815,False,12.020815,0.0
3,0,2.5,3.25,0.0,3,2.5,11.75,0.0,8.500000,True,8.500000,0.0
4,0,2.5,3.25,0.0,4,2.5,3.25,11.0,11.000000,True,0.000000,11.0
5,0,2.5,3.25,0.0,5,-6.0,3.25,11.0,13.901439,False,8.500000,11.0
6,0,2.5,3.25,0.0,6,-6.0,11.75,11.0,16.294171,False,12.020815,11.0
7,0,2.5,3.25,0.0,7,2.5,11.75,11.0,13.901439,False,8.500000,11.0
8,0,2.5,3.25,0.0,8,-3.0,5.00,11.0,12.422258,False,5.771698,11.0
9,0,2.5,3.25,0.0,9,-3.0,10.00,11.0,14.028988,False,8.707037,11.0
10,0,2.5,3.25,0.0,10,2.0,10.00,11.0,12.915591,False,6.768493,11.0


In [346]:
edificio_pairs = edificio_pairs[edificio_pairs['are_connected']==True]

edificio_pairs.drop(columns=['are_connected'], inplace=True)

edificio_pairs.head(10)

,index1,x_x,y_x,z_x,index2,x_y,y_y,z_y,euclidean_distance,d_horizontal,d_vertical
1,0,2.5,3.25,0.0,1,-6.0,3.25,0.0,8.500000,8.500000,0.0
3,0,2.5,3.25,0.0,3,2.5,11.75,0.0,8.500000,8.500000,0.0
4,0,2.5,3.25,0.0,4,2.5,3.25,11.0,11.000000,0.000000,11.0
18,0,2.5,3.25,0.0,18,-1.5,3.50,0.0,4.007805,4.007805,0.0
19,0,2.5,3.25,0.0,19,1.5,3.50,0.0,1.030776,1.030776,0.0
64,1,-6.0,3.25,0.0,0,2.5,3.25,0.0,8.500000,8.500000,0.0
66,1,-6.0,3.25,0.0,2,-6.0,11.75,0.0,8.500000,8.500000,0.0
69,1,-6.0,3.25,0.0,5,-6.0,3.25,11.0,11.000000,0.000000,11.0
82,1,-6.0,3.25,0.0,18,-1.5,3.50,0.0,4.506939,4.506939,0.0
83,1,-6.0,3.25,0.0,19,1.5,3.50,0.0,7.504166,7.504166,0.0


# Computing time travel

In [347]:
d_horizontal = edificio_pairs['d_horizontal']
d_vertical = edificio_pairs['d_vertical']
rows_horizontal = (d_vertical == 0)
rows_ascending = (d_horizontal == 0) & (d_vertical > 0)
rows_descending = (d_horizontal == 0) & (d_vertical < 0)
rows_oblique_ascending = (d_horizontal != 0) & (d_vertical > 0)
rows_oblique_descending = (d_horizontal != 0) & (d_vertical < 0)

In [348]:
def compute_time_travel(dataframe):
    dataframe.loc[rows_horizontal, 'time_travel'] = dataframe.loc[rows_horizontal, 'd_horizontal'] / HORIZONTAL_SPEED

    dataframe.loc[rows_ascending, 'time_travel'] = dataframe.loc[rows_ascending, 'd_vertical'] / ASCENDING_SPEED
    dataframe.loc[rows_descending, 'time_travel'] = -dataframe.loc[rows_descending, 'd_vertical'] / DESCENDING_SPEED

    dataframe.loc[rows_oblique_ascending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_ascending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     dataframe.loc[rows_oblique_ascending, 'd_vertical'] / ASCENDING_SPEED)
    
    dataframe.loc[rows_oblique_descending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_descending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     -dataframe.loc[rows_oblique_descending, 'd_vertical'] / DESCENDING_SPEED)
    return dataframe

# Computing energy consumption


In [349]:
def compute_energy_cost(dataframe):
    dataframe.loc[rows_horizontal, 'energy_consumption'] = dataframe.loc[rows_horizontal, 'd_horizontal'] * HORIZONTAL_ENERGY

    dataframe.loc[rows_ascending, 'energy_consumption'] = dataframe.loc[rows_ascending, 'd_vertical'] * ASCENDING_ENERGY
    dataframe.loc[rows_descending, 'energy_consumption'] = -dataframe.loc[rows_descending, 'd_vertical'] * DESCENDING_ENERGY

    dataframe.loc[rows_oblique_ascending, 'energy_consumption'] = (dataframe.loc[rows_oblique_ascending, 'd_horizontal'] * HORIZONTAL_ENERGY +
                                                                  dataframe.loc[rows_oblique_ascending, 'd_vertical'] * ASCENDING_ENERGY)
    
    dataframe.loc[rows_oblique_descending, 'energy_consumption'] = (dataframe.loc[rows_oblique_descending, 'd_horizontal'] * HORIZONTAL_ENERGY +
                                                                  -dataframe.loc[rows_oblique_descending, 'd_vertical'] * DESCENDING_ENERGY)
    return dataframe

In [350]:
edificio_pairs = compute_time_travel(edificio_pairs)

edificio_pairs = compute_energy_cost(edificio_pairs)

edificio_pairs.head(10)


,index1,x_x,y_x,z_x,index2,x_y,y_y,z_y,euclidean_distance,d_horizontal,d_vertical,time_travel,energy_consumption
1,0,2.5,3.25,0.0,1,-6.0,3.25,0.0,8.500000,8.500000,0.0,5.666667,85.000000
3,0,2.5,3.25,0.0,3,2.5,11.75,0.0,8.500000,8.500000,0.0,5.666667,85.000000
4,0,2.5,3.25,0.0,4,2.5,3.25,11.0,11.000000,0.000000,11.0,11.000000,550.000000
18,0,2.5,3.25,0.0,18,-1.5,3.50,0.0,4.007805,4.007805,0.0,2.671870,40.078049
19,0,2.5,3.25,0.0,19,1.5,3.50,0.0,1.030776,1.030776,0.0,0.687184,10.307764
64,1,-6.0,3.25,0.0,0,2.5,3.25,0.0,8.500000,8.500000,0.0,5.666667,85.000000
66,1,-6.0,3.25,0.0,2,-6.0,11.75,0.0,8.500000,8.500000,0.0,5.666667,85.000000
69,1,-6.0,3.25,0.0,5,-6.0,3.25,11.0,11.000000,0.000000,11.0,11.000000,550.000000
82,1,-6.0,3.25,0.0,18,-1.5,3.50,0.0,4.506939,4.506939,0.0,3.004626,45.069391
83,1,-6.0,3.25,0.0,19,1.5,3.50,0.0,7.504166,7.504166,0.0,5.002777,75.041655
